# 4. Orquestación de Agentes con CrewAI

## Objetivos de Aprendizaje
- Entender el concepto de agentes colaborativos y los roles en un "equipo" (Crew).
- Aprender los componentes clave de CrewAI: `Agent`, `Task`, `Tool` y `Crew`.
- Definir un equipo de agentes (un investigador y un escritor) para realizar una tarea compleja.
- Ejecutar el `Crew` y observar cómo los agentes colaboran y se pasan el trabajo entre ellos.
- **Comprender las configuraciones específicas necesarias para usar CrewAI con GitHub Models API**.

## ¿Qué es CrewAI y por qué usarlo?

Mientras que LangChain proporciona los bloques de construcción fundamentales para crear un agente, **CrewAI** se especializa en la **orquestación de múltiples agentes autónomos**. La idea central es que, para resolver tareas complejas, es más eficiente tener un equipo de agentes especializados que colaboren, en lugar de un solo agente que intente hacerlo todo.

**Analogía:** Piensa en una agencia de marketing. No tienes una sola persona que es experta en investigación, redacción, diseño y redes sociales. Tienes un equipo donde cada miembro tiene un rol claro. CrewAI aplica este concepto a los agentes de IA.

**Ventajas de CrewAI:**
- **Roles Especializados**: Permite definir agentes con roles, objetivos (`goal`) e historias de fondo (`backstory`) específicas, lo que los hace más efectivos en su nicho.
- **Colaboración Autónoma**: Los agentes pueden delegar tareas entre ellos de forma autónoma.
- **Procesos Secuenciales y Jerárquicos**: Soporta flujos de trabajo donde las tareas se completan en un orden específico.
- **Claridad y Estructura**: El código es muy declarativo y fácil de leer, ya que se centra en definir el equipo y sus responsabilidades.

## Relación entre CrewAI y LangChain

CrewAI **utiliza LangChain internamente** para manejar los LLMs y las herramientas. Esto significa que:

1. **CrewAI** se encarga de la orquestación de agentes (el "director de orquesta")
2. **LangChain** proporciona la interfaz con los modelos y herramientas (los "instrumentos")

Esta relación requiere configuraciones específicas que veremos en este notebook.

### 1. Instalación y Configuración

In [1]:
!pip install crewai langchain-openai openai wikipedia -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\guts\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### 2. Configuración Especial para GitHub Models API

**⚠️ IMPORTANTE:** Esta es la parte crítica que causa problemas si no se configura correctamente.

**El Problema:**
- CrewAI utiliza LangChain internamente
- LangChain busca las variables de entorno `OPENAI_API_KEY` y `OPENAI_API_BASE`
- Nosotros tenemos `GITHUB_TOKEN` y `OPENAI_BASE_URL`
- Necesitamos "mapear" nuestras variables a las que LangChain espera

**La Solución:**
Configuramos las variables de entorno que LangChain espera, usando nuestros valores de GitHub Models API.

In [4]:
import os

# 🔧 CONFIGURACIÓN CRÍTICA: Mapear variables de entorno para LangChain
# CrewAI → LangChain → GitHub Models API

# LangChain espera estas variables específicas:
os.environ["OPENAI_API_BASE"] = os.environ.get("OPENAI_BASE_URL", "")
os.environ["OPENAI_API_KEY"] = os.environ.get("GITHUB_TOKEN", "")

# Verificar que las variables estén configuradas
print("🔍 Verificando configuración:")
print(f"OPENAI_BASE_URL: {os.environ.get('OPENAI_BASE_URL', 'No configurado')}")
print(f"GITHUB_TOKEN: {'✅ Configurado' if os.environ.get('GITHUB_TOKEN') else '❌ No configurado'}")
print(f"OPENAI_API_BASE: {os.environ.get('OPENAI_API_BASE', 'No configurado')}")
print(f"OPENAI_API_KEY: {'✅ Configurado' if os.environ.get('OPENAI_API_KEY') else '❌ No configurado'}")

print("\n✅ Variables de entorno mapeadas correctamente para LangChain.")

🔍 Verificando configuración:
OPENAI_BASE_URL: https://models.inference.ai.azure.com
GITHUB_TOKEN: ✅ Configurado
OPENAI_API_BASE: https://models.inference.ai.azure.com
OPENAI_API_KEY: ✅ Configurado

✅ Variables de entorno mapeadas correctamente para LangChain.


### 3. Configuración del LLM

Ahora configuramos el LLM usando LangChain. Gracias a la configuración anterior, LangChain automáticamente usará nuestras variables de entorno mapeadas.

In [5]:
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang('es')

# 🧠 Configurar el LLM con LangChain
# Nota: No necesitamos pasar explícitamente las API keys aquí
# porque LangChain las lee automáticamente de las variables de entorno
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0
    )
    
    # Probar que el LLM funciona
    test_response = llm.invoke("Hola, ¿funcionas correctamente?")
    print(f"✅ LLM configurado y probado exitosamente.")
    print(f"📝 Respuesta de prueba: {test_response.content[:50]}...")
    
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    print("💡 Verifica que OPENAI_BASE_URL y GITHUB_TOKEN estén configurados correctamente.")
    llm = None

✅ LLM configurado y probado exitosamente.
📝 Respuesta de prueba: ¡Hola! Sí, funciono correctamente. 😊 ¿En qué puedo...


### 4. Definición de Herramientas con CrewAI

**⚠️ IMPORTANTE:** CrewAI requiere un enfoque específico para las herramientas.

**El Problema:**
- LangChain usa el decorador `@tool` para definir herramientas
- CrewAI requiere que las herramientas hereden de `BaseTool`
- Mezclar ambos enfoques causa errores

**La Solución:**
Usar `BaseTool` de `crewai_tools` para crear herramientas compatibles con CrewAI.

In [6]:
from crewai.tools import BaseTool

# 🔧 HERRAMIENTA CORREGIDA: Usar BaseTool en lugar de @tool
class WikipediaSearchTool(BaseTool):
    name: str = "Wikipedia Search Tool"
    description: str = "Busca en Wikipedia un tema y devuelve un resumen detallado. Es ideal para obtener información sobre personas, lugares, conceptos históricos y científicos."
    
    def _run(self, query: str) -> str:
        """Ejecuta la búsqueda en Wikipedia"""
        try:
            # Configurar el idioma de Wikipedia
            wikipedia.set_lang("es")
            # Devolver un resumen detallado para que el escritor tenga más material
            return wikipedia.summary(query, sentences=5)
        except wikipedia.exceptions.PageError:
            return f"No se encontró ninguna página para '{query}'. Intenta con un término más específico."
        except wikipedia.exceptions.DisambiguationError as e:
            return f"La búsqueda para '{query}' es ambigua. Opciones disponibles: {e.options[:3]}. Especifica cuál te interesa."
        except Exception as e:
            return f"Error al buscar en Wikipedia: {str(e)}"

# Crear la instancia de la herramienta
wikipedia_tool = WikipediaSearchTool()
tools = [wikipedia_tool]

# Probar la herramienta
try:
    test_result = wikipedia_tool._run("Albert Einstein")
    print("✅ Herramienta de Wikipedia configurada y probada exitosamente.")
    print(f"📝 Resultado de prueba: {test_result[:100]}...")
except Exception as e:
    print(f"❌ Error probando la herramienta: {e}")

print(f"\n🔧 {len(tools)} herramienta(s) disponible(s) para los agentes.")

✅ Herramienta de Wikipedia configurada y probada exitosamente.
📝 Resultado de prueba: Albert Einstein pronunciación en alemán: /ˈalbɐt ˈaɪnʃtaɪn/ ();​ (Ulm, 14 de marzo de 1879-Princeton...

🔧 1 herramienta(s) disponible(s) para los agentes.


### 5. Creación del Equipo de Agentes (Crew)

Ahora definimos nuestro equipo de agentes especializados:

1. **Investigador (Researcher)**: Busca información detallada usando Wikipedia
2. **Escritor (Writer)**: Transforma la información en una biografía bien redactada

**⚠️ IMPORTANTE:** Otro error común es el parámetro `verbose` en `Crew`.

In [13]:
from crewai import Agent, Task, Crew, Process

# 🕵️ Agente 1: El Investigador
researcher = Agent(
    role="Investigador Senior",
    goal="Encontrar información completa y precisa sobre personas históricas, científicos y figuras importantes utilizando fuentes confiables.",
    backstory="""Eres un investigador académico con años de experiencia en la búsqueda de información histórica y científica. 
    Tu especialidad es encontrar datos precisos y relevantes en Wikipedia y otras fuentes confiables. 
    Te enorgulleces de la exactitud de tu trabajo y siempre proporcionas contexto histórico relevante. 
    No escribes biografías completas, tu trabajo es recopilar los datos más importantes y precisos.""",
    tools=tools,
    llm="openai/gpt-4o-mini",
    verbose=True,
    allow_delegation=False  # Este agente no delega trabajo
)

# ✍️ Agente 2: El Escritor
writer = Agent(
    role="Escritor de Biografías",
    goal="Crear biografías atractivas, bien estructuradas y fáciles de leer basadas en la información proporcionada por el investigador.",
    backstory="""Eres un escritor profesional especializado en biografías y divulgación científica. 
    Tu habilidad única es transformar datos técnicos y históricos en narrativas cautivadoras que son 
    tanto informativas como accesibles para el público general. 
    Tienes un don especial para destacar los aspectos más interesantes de la vida de las personas 
    y presentar sus logros de manera inspiradora.""",
    llm="openai/gpt-4o-mini",
    verbose=True,
    allow_delegation=False
)

print("✅ Agentes creados exitosamente:")
print(f"🕵️ {researcher.role}")
print(f"✍️ {writer.role}")

✅ Agentes creados exitosamente:
🕵️ Investigador Senior
✍️ Escritor de Biografías


### 6. Definición de Tareas

Las tareas definen exactamente qué debe hacer cada agente y cómo se relacionan entre ellas.

In [14]:
# 🔍 Tarea 1: Investigación
research_task = Task(
    description="""Busca información detallada sobre Marie Curie en Wikipedia. 
    Enfócate en:
    - Sus descubrimientos científicos más importantes
    - Su impacto en la ciencia y la sociedad
    - Datos biográficos clave (fechas, lugares, educación)
    - Sus premios y reconocimientos
    - Su legado científico
    
    Proporciona información precisa y bien organizada que el escritor pueda usar.""",
    expected_output="Un resumen detallado de 4-6 párrafos con los datos más importantes sobre la vida, descubrimientos y legado de Marie Curie.",
    agent=researcher
)

# ✍️ Tarea 2: Escritura
write_task = Task(
    description="""Usando la información recopilada por el investigador, escribe una biografía cautivadora de Marie Curie.
    
    Requisitos:
    - Mínimo 5 párrafos bien estructurados
    - Estilo atractivo y accesible
    - Incluir sus logros más importantes
    - Destacar su impacto en la ciencia
    - Formato Markdown con encabezados apropiados
    - Tono inspirador pero preciso
    
    La biografía debe ser educativa e inspiradora para lectores de todas las edades.""",
    expected_output="Una biografía completa en formato Markdown, bien estructurada y atractiva, de al menos 5 párrafos.",
    agent=writer,
    context=[research_task]  # Esta tarea depende del resultado de la investigación
)

print("✅ Tareas definidas exitosamente:")
print(f"🔍 Tarea de investigación: {research_task.description[:50]}...")
print(f"✍️ Tarea de escritura: {write_task.description[:50]}...")

✅ Tareas definidas exitosamente:
🔍 Tarea de investigación: Busca información detallada sobre Marie Curie en W...
✍️ Tarea de escritura: Usando la información recopilada por el investigad...


### 7. Ensamblaje del Equipo (Crew)

**⚠️ CONFIGURACIÓN CORREGIDA:** El parámetro `verbose` debe ser boolean, no entero.

In [15]:
# 🎯 CONFIGURACIÓN CORREGIDA: verbose debe ser boolean
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,  # Las tareas se ejecutan en orden
    verbose=True  # ✅ CORRECTO: boolean, no entero (verbose=2 causaría error)
)

print("✅ Equipo (Crew) ensamblado exitosamente:")
print(f"👥 {len(crew.agents)} agentes en el equipo")
print(f"📋 {len(crew.tasks)} tareas definidas")
print(f"🔄 Proceso: {crew.process}")
print(f"🔊 Verbose: {crew.verbose}")

✅ Equipo (Crew) ensamblado exitosamente:
👥 2 agentes en el equipo
📋 2 tareas definidas
🔄 Proceso: Process.sequential
🔊 Verbose: True


### 8. Ejecución del Crew

¡Ahora viene la magia! Ejecutamos el crew y observamos cómo los agentes colaboran.

In [20]:
from crewai.tools import BaseTool

print("Cantidad de tools:", len(tools))

for i, tool in enumerate(tools):
    print(f"Tool {i}:")
    print("Tipo:", type(tool))
    print("Es BaseTool:", isinstance(tool, BaseTool))
    print("Nombre:", getattr(tool, "name", None))
    print("Descripción:", getattr(tool, "description", None))

Cantidad de tools: 1
Tool 0:
Tipo: <class '__main__.WikipediaSearchTool'>
Es BaseTool: True
Nombre: Wikipedia Search Tool
Descripción: Tool Name: wikipedia_search_tool
Tool Arguments: {
  "properties": {
    "query": {
      "title": "Query",
      "type": "string"
    }
  },
  "required": [
    "query"
  ],
  "title": "WikipediaSearchToolSchema",
  "type": "object",
  "additionalProperties": false
}
Tool Description: Busca en Wikipedia un tema y devuelve un resumen detallado. Es ideal para obtener información sobre personas, lugares, conceptos históricos y científicos.


In [17]:
import os
from crewai import LLM

llm = LLM(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.github.ai/inference"
)

print("✅ LLM configurado con GitHub Models")

✅ LLM configurado con GitHub Models


In [ ]:
result = crew.kickoff()
print(result)

In [22]:
# 🚀 Ejecutar el crew
if llm is None:
    print("❌ No se puede ejecutar el crew sin un LLM configurado.")
    print("💡 Verifica la configuración de las variables de entorno en las celdas anteriores.")
else:
    try:
        print("🚀 Iniciando ejecución del crew...")
        print("📝 Observa cómo los agentes colaboran paso a paso:\n")
        
        # Testing para ver si recibe bien los parametros
        print("llm:", type(llm))
        print("tools:", type(tools))
        print("researcher:", type(researcher))
        print("writer:", type(writer))
        print("crew:", type(crew))

        # Ejecutar el crew
        result = await crew.kickoff_async()
        
        print("\n" + "="*80)
        print("🏁 RESULTADO FINAL DEL CREW")
        print("="*80)
        print(result)
        
    except Exception as e:
        print(f"❌ Error durante la ejecución del crew: {e}")
        print("\n🔧 Posibles soluciones:")
        print("1. Verifica que GITHUB_TOKEN esté configurado correctamente")
        print("2. Asegúrate de que OPENAI_BASE_URL esté configurado")
        print("3. Confirma que tu token tenga permisos para usar GitHub Models")
        print("4. Verifica que no hayas mezclado @tool con BaseTool")
        print("5. Asegúrate de que verbose=True (no verbose=2)")
        
        # Mostrar información de debugging
        print("\n🐛 Información de debugging:")
        import traceback
        traceback.print_exc()

🚀 Iniciando ejecución del crew...
📝 Observa cómo los agentes colaboran paso a paso:

llm: <class 'crewai.llms.providers.openai.completion.OpenAICompletion'>
tools: <class 'list'>
researcher: <class 'crewai.agent.core.Agent'>
writer: <class 'crewai.agent.core.Agent'>
crew: <class 'crewai.crew.Crew'>


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7056327c-5b22-4080-a0e0-5c634edffb1c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Busca información detallada sobre Marie Curie en Wikipedia.                                              │
│      Enfócate en:                                                                                               │
│      - Sus descubrimientos científicos más importantes                                                          │
│      - Su impacto en la ciencia y la sociedad                                                                   │
│      - Datos biográficos clave (fechas, lugares, educación)                                                     │
│      - Sus premios y reconocimientos                                                                            │
│      - Su legado científico                                                                                     │
│                                                                                                                 │
│      Proporciona información precisa y bien organizada que el escritor pueda usar.                              │
│  ID: 0d5d4dd0-b09f-4c90-b9e8-3784da9adab4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador Senior                                                                                     │
│                                                                                                                 │
│  Task: Busca información detallada sobre Marie Curie en Wikipedia.                                              │
│      Enfócate en:                                                                                               │
│      - Sus descubrimientos científicos más importantes                                                          │
│      - Su impacto en la ciencia y la sociedad                                                                   │
│      - Datos biográficos clave (fechas, lugares, educación)                                                     │
│      - Sus premios y reconocimientos                                                                            │
│      - Su legado científico                                                                                     │
│                                                                                                                 │
│      Proporciona información precisa y bien organizada que el escritor pueda usar.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: wikipedia_search_tool                                                                                    │
│  Args: {'query': 'Marie Curie'}                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Escritor de Biografías                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Biografía de Marie Curie: Pionera de la Ciencia                                                              │
│                                                                                                                 │
│  Marie Curie, nacida Maria Salomea Skłodowska el 7 de noviembre de 1867 en Varsovia, Polonia, no solo es        │
│  recordada como una científica excepcional, sino como un símbolo de la tenacidad y el espíritu indomable que    │
│  impulsan el avance del conocimiento humano. Su vida estuvo marcada por un deseo insaciable de aprender, lo     │
│  que la llevó a desafiar las normas restrictivas de su país bajo el dominio ruso. A pesar de las limitaciones   │
│  educativas de su tiempo, se formó en una "universidad flotante", un innovador concepto que buscaba             │
│  proporcionar educación a las mujeres y otros estudiantes que no podían acceder a la educación formal. Este     │
│  primer paso, lleno de dificultad y determinación, sentó las bases de una carrera que cambiaría el mundo.       │
│                                                                                                                 │
│  El legado científico de Curie es extraordinario. Pionera en el estudio de la radiactividad, fue la primera     │
│  persona en recibir dos premios Nobel en distintas disciplinas científicas. En 1903, recibió el Premio Nobel    │
│  de Física, compartido con su esposo Pierre Curie y el físico Henri Becquerel, por sus aportes fundamentales    │
│  al entendimiento de la radiactividad. Ocho años después, en 1911, hizo historia nuevamente al recibir el       │
│  Premio Nobel de Química por el descubrimiento de los elementos radiactivos polonio y radio. Su arduo trabajo,  │
│  combinando la pasión con la investigación metódica, no solo redefinió la física y la química, sino que         │
│  también abrió nuevas vías de exploración científica.                                                           │
│                                                                                                                 │
│  Los descubrimientos de Curie tuvieron un impacto profundo en el campo de la medicina, especialmente en la      │
│  radioterapia para el tratamiento del cáncer. La radiactividad, término que ella misma acuñó, permitió el       │
│  desarrollo de nuevas técnicas de diagnóstico y tratamiento que siguen en uso en la actualidad. A medida que    │
│  sus hallazgos continuaron siendo explorados, el mundo médico se benefició enormemente, marcando un hito en la  │
│  forma en que se abordaban enfermedades anteriormente incurables. El trabajo de Curie no solo salvó vidas,      │
│  sino que dejó una huella imborrable en la historia de la ciencia.                                              │
│                                                                                                                 │
│  Más allá de sus logros científicos, Marie Curie también fue una pionera en la lucha por la igualdad de género  │
│  en la ciencia. Como primera mujer en ocupar el puesto de profesora en la Universidad de París, desafió las     │
│  convenciones de su época, creando un precedente importante para futuras generaciones de mujeres en el ámbito   │
│  científico. Su entrada triunfal en un mundo predominantemente masculino no solo rompió barreras, sino que      │
│  también abrazó la idea de que el talento no conoce de 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Usando la información recopilada por el investigador, escribe una biografía cautivadora de Marie Curie.  │
│                                                                                                                 │
│      Requisitos:                                                                                                │
│      - Mínimo 5 párrafos bien estructurados                                                                     │
│      - Estilo atractivo y accesible                                                                             │
│      - Incluir sus logros más importantes                                                                       │
│      - Destacar su impacto en la ciencia                                                                        │
│      - Formato Markdown con encabezados apropiados                                                              │
│      - Tono inspirador pero preciso                                                                             │
│                                                                                                                 │
│      La biografía debe ser educativa e inspiradora para lectores de todas las edades.                           │
│  Agent: Escritor de Biografías                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7056327c-5b22-4080-a0e0-5c634edffb1c                                                                       │
│  Final Output: # Biografía de Marie Curie: Pionera de la Ciencia                                                │
│                                                                                                                 │
│  Marie Curie, nacida Maria Salomea Skłodowska el 7 de noviembre de 1867 en Varsovia, Polonia, no solo es        │
│  recordada como una científica excepcional, sino como un símbolo de la tenacidad y el espíritu indomable que    │
│  impulsan el avance del conocimiento humano. Su vida estuvo marcada por un deseo insaciable de aprender, lo     │
│  que la llevó a desafiar las normas restrictivas de su país bajo el dominio ruso. A pesar de las limitaciones   │
│  educativas de su tiempo, se formó en una "universidad flotante", un innovador concepto que buscaba             │
│  proporcionar educación a las mujeres y otros estudiantes que no podían acceder a la educación formal. Este     │
│  primer paso, lleno de dificultad y determinación, sentó las bases de una carrera que cambiaría el mundo.       │
│                                                                                                                 │
│  El legado científico de Curie es extraordinario. Pionera en el estudio de la radiactividad, fue la primera     │
│  persona en recibir dos premios Nobel en distintas disciplinas científicas. En 1903, recibió el Premio Nobel    │
│  de Física, compartido con su esposo Pierre Curie y el físico Henri Becquerel, por sus aportes fundamentales    │
│  al entendimiento de la radiactividad. Ocho años después, en 1911, hizo historia nuevamente al recibir el       │
│  Premio Nobel de Química por el descubrimiento de los elementos radiactivos polonio y radio. Su arduo trabajo,  │
│  combinando la pasión con la investigación metódica, no solo redefinió la física y la química, sino que         │
│  también abrió nuevas vías de exploración científica.                                                           │
│                                                                                                                 │
│  Los descubrimientos de Curie tuvieron un impacto profundo en el campo de la medicina, especialmente en la      │
│  radioterapia para el tratamiento del cáncer. La radiactividad, término que ella misma acuñó, permitió el       │
│  desarrollo de nuevas técnicas de diagnóstico y tratamiento que siguen en uso en la actualidad. A medida que    │
│  sus hallazgos continuaron siendo explorados, el mundo médico se benefició enormemente, marcando un hito en la  │
│  forma en que se abordaban enfermedades anteriormente incurables. El trabajo de Curie no solo salvó vidas,      │
│  sino que dejó una huella imborrable en la historia de la ciencia.                                              │
│                                                                                                                 │
│  Más allá de sus logros científicos, Marie Curie también fue una pionera en la lucha por la igualdad de género  │
│  en la ciencia. Como primera mujer en ocupar el puesto de profesora en la Universidad de París, desafió las     │
│  convenciones de su época, creando un precedente importante para futuras generaciones de mujeres en el ámbito   │
│  científico. Su entrada triunfal en un mundo predominantemente masculino no solo rompió barreras, sino que      │
│  también abrazó la idea de que el talento no conoce de


🏁 RESULTADO FINAL DEL CREW
# Biografía de Marie Curie: Pionera de la Ciencia

Marie Curie, nacida Maria Salomea Skłodowska el 7 de noviembre de 1867 en Varsovia, Polonia, no solo es recordada como una científica excepcional, sino como un símbolo de la tenacidad y el espíritu indomable que impulsan el avance del conocimiento humano. Su vida estuvo marcada por un deseo insaciable de aprender, lo que la llevó a desafiar las normas restrictivas de su país bajo el dominio ruso. A pesar de las limitaciones educativas de su tiempo, se formó en una "universidad flotante", un innovador concepto que buscaba proporcionar educación a las mujeres y otros estudiantes que no podían acceder a la educación formal. Este primer paso, lleno de dificultad y determinación, sentó las bases de una carrera que cambiaría el mundo.

El legado científico de Curie es extraordinario. Pionera en el estudio de la radiactividad, fue la primera persona en recibir dos premios Nobel en distintas disciplinas científicas.



┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                                             │
└─────────────────────────────────────

## 🎓 Resumen de Configuraciones Críticas

### Problemas Comunes y Sus Soluciones

| **Problema** | **Síntoma** | **Solución** |
|-------------|-------------|-------------|
| **Variables de entorno** | `AuthenticationError: Incorrect API key` | Mapear `GITHUB_TOKEN` → `OPENAI_API_KEY` y `OPENAI_BASE_URL` → `OPENAI_API_BASE` |
| **Herramientas incompatibles** | `'Tool' object is not callable` | Usar `BaseTool` en lugar de `@tool` |
| **Parámetro verbose** | `ValidationError: Input should be a valid boolean` | Usar `verbose=True` en lugar de `verbose=2` |
| **LLM no configurado** | `llm is None` | Verificar que las variables de entorno estén configuradas |

### Configuración Correcta para GitHub Models API

```python
# 1. Mapear variables de entorno
os.environ["OPENAI_API_BASE"] = os.environ.get("OPENAI_BASE_URL", "")
os.environ["OPENAI_API_KEY"] = os.environ.get("GITHUB_TOKEN", "")

# 2. Configurar LLM sin parámetros explícitos
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 3. Usar BaseTool para herramientas
class MiTool(BaseTool):
    name: str = "Mi Herramienta"
    description: str = "Descripción de la herramienta"
    def _run(self, query: str) -> str:
        return "resultado"

# 4. Configurar Crew con verbose boolean
crew = Crew(
    agents=[agent1, agent2],
    tasks=[task1, task2],
    process=Process.sequential,
    verbose=True  # ✅ boolean, no entero
)
```

### Variables de Entorno Requeridas

```bash
export OPENAI_BASE_URL="https://models.inference.ai.azure.com"
export GITHUB_TOKEN="tu_token_de_github_aquí"
```

## 🚀 Conclusiones

### Lo que hemos aprendido:

1. **CrewAI vs LangChain**: CrewAI orquesta equipos de agentes, LangChain proporciona los componentes individuales.

2. **Configuración crítica**: Cuando usas CrewAI con GitHub Models API, necesitas mapear las variables de entorno correctamente.

3. **Herramientas compatibles**: CrewAI requiere `BaseTool`, no el decorador `@tool` de LangChain.

4. **Parámetros correctos**: `verbose` debe ser boolean, no entero.

5. **Colaboración de agentes**: Los agentes pueden trabajar en secuencia, pasándose información entre ellos.

### Próximos pasos:

En los siguientes módulos exploraremos:
- Sistemas de memoria para agentes
- Integración con herramientas externas
- Estrategias de planificación más avanzadas
- Observabilidad y monitoreo de agentes

### 💡 Tip para estudiantes:

**Siempre revisa las configuraciones de entorno cuando cambies entre frameworks.** Cada framework tiene sus propias convenciones y expectativas sobre cómo acceder a los servicios externos. La clave está en entender estas diferencias y configurar correctamente las interfaces entre ellos.